# Fogli & Veldkamp Replication with bayespecon

This notebook replicates the **data construction and core spatial-panel estimation workflow** in:
`reference/Supplement-Data-and-Matlab-code-Fogli-and-Veldkamp-Replication/ReplicationFinalPaper.m`

Using the `bayespecon` package, we estimate the closest panel spatial specifications available in this codebase:
- `SDMPanelFE` (spatial lag + spatially lagged covariates)
- `SDEMPanelFE` (spatial error + spatially lagged covariates)
- `SARPanelFE` (spatial lag without spatially lagged covariates)

The original MATLAB routine is a dynamic spatial panel with interactive fixed effects (`SFactors`), which is not currently implemented in `bayespecon`.

In [1]:
from pathlib import Path
import sys

import arviz as az
import numpy as np
import pandas as pd
from scipy.io import loadmat

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from bayespecon import SDMPanelFE, SDEMPanelFE, SARPanelFE

az.style.use("arviz-white")

ref_dir = repo_root / "reference" / "Supplement-Data-and-Matlab-code-Fogli-and-Veldkamp-Replication"
xls_path = ref_dir / "Data Fogli Veldkamp.xls"
mat_path = ref_dir / "WBC.mat"

print("Reference directory:", ref_dir)
print("Excel exists:", xls_path.exists())
print("WBC.mat exists:", mat_path.exists())

Reference directory: /Users/knaaptime/Dropbox/projects/bayespecon/reference/Supplement-Data-and-Matlab-code-Fogli-and-Veldkamp-Replication
Excel exists: True
WBC.mat exists: True


In [2]:
# Column map from MATLAB (1-based -> 0-based)
COLS = {
    "lfp": 2,
    "pc_urb": 3,
    "pc_farm": 4,
    "avg_educ": 9,
    "pop_dens": 10,
    "wages": 15,
    "xcor": 17,
    "ycor": 18,
}

# The source .xls includes a text header row; keep all numeric rows (including NaNs in data).
df_raw = pd.read_excel(xls_path, header=None)
df_num = df_raw.apply(pd.to_numeric, errors="coerce")
df_num = df_num[df_num.iloc[:, 0].notna()].reset_index(drop=True)
A = df_num.to_numpy(dtype=float)
WBC = loadmat(mat_path)["WBC"].astype(float)

print("A shape:", A.shape)
print("WBC shape:", WBC.shape)

A shape: (21518, 19)
WBC shape: (3074, 3074)


In [3]:
def row_normalize(W):
    W = W.copy().astype(float)
    rs = W.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1.0
    return W / rs


def build_knn_from_coords(x, y, k=6):
    coords = np.column_stack([x, y])
    n = coords.shape[0]
    W = np.zeros((n, n), dtype=float)
    for i in range(n):
        d = np.sum((coords - coords[i]) ** 2, axis=1)
        idx = np.argsort(d)[1:k+1]  # skip self
        W[i, idx] = 1.0 / k
    return W


def make_dynamic_panel_df(y_vec, x_mat, W, feature_names):
    """Build period t>=2 dataset with lagged dependent variable."""
    nobs = y_vec.shape[0]
    N = W.shape[0]
    T = nobs // N

    y = y_vec.reshape(T, N)
    X = x_mat.reshape(T, N, x_mat.shape[1])

    # MATLAB constructs Wy period-by-period
    Wy = np.stack([W @ y[t] for t in range(T)], axis=0)

    rows = []
    for t in range(1, T):
        for i in range(N):
            row = {
                "unit": i,
                "time": t,
                "lfp": float(y[t, i]),
                "lfp_lag": float(y[t-1, i]),
                "w_lfp_lag": float(Wy[t-1, i]),
            }
            for j, nm in enumerate(feature_names):
                row[nm] = float(X[t, i, j])
            rows.append(row)

    return pd.DataFrame(rows), T

## Spec A (MATLAB first block): wages included, 6-nearest-neighbor W, balanced sample N=1569

In [4]:
N1 = 3074
T1 = 6

xcor = A[:N1, COLS["xcor"]]
ycor = A[:N1, COLS["ycor"]]

# Same missingness logic as MATLAB on rows N1+1:end and variables [3,4,5,10,11,16]
check_cols = [COLS["lfp"], COLS["pc_urb"], COLS["pc_farm"], COLS["avg_educ"], COLS["pop_dens"], COLS["wages"]]
M1 = np.sum(np.isnan(A[N1:, check_cols]), axis=1)
M2 = M1.reshape(N1, T1, order="F")
select = (np.sum(M2 > 0, axis=1) == 0)
N_a = int(np.sum(select))

W_a = build_knn_from_coords(xcor[select], ycor[select], k=6)

# Select full 7-period stacked panel rows
selectfull = np.tile(select, 7)
y_a = A[selectfull, COLS["lfp"]]
x_a = A[selectfull][:, [COLS["pc_urb"], COLS["pc_farm"], COLS["avg_educ"], COLS["pop_dens"], COLS["wages"]]]
x_a[:, 4] = x_a[:, 4] / 1000.0  # same wage scaling as MATLAB

feature_names_a = ["pc_urb", "pc_farm", "avg_educ", "pop_dens", "wages_k"]
df_a, T_a = make_dynamic_panel_df(y_a, x_a, W_a, feature_names_a)

print("Spec A units:", N_a)
print("Spec A periods in transformed panel:", T_a - 1)
display(df_a.head())

Spec A units: 1569
Spec A periods in transformed panel: 6


,unit,time,lfp,lfp_lag,w_lfp_lag,pc_urb,pc_farm,avg_educ,pop_dens,wages_k
0,0,1,25.722452,22.180399,26.799580,24.111956,51.209721,7.1,30.0,7.333390
1,1,1,32.488224,27.390698,25.304910,23.902811,46.819187,6.4,32.0,6.774958
2,2,1,13.164938,11.679274,28.358566,0.000000,40.290211,7.0,29.0,4.191670
3,3,1,14.077820,10.115013,19.320397,9.670405,69.884384,7.3,45.0,5.772629
4,4,1,27.605156,27.011340,23.777468,23.200356,46.872860,6.9,38.0,6.216501


In [6]:
formula_a = "lfp ~ lfp_lag + w_lfp_lag + pc_urb + pc_farm + avg_educ + pop_dens + wages_k"

# Closest counterpart to MATLAB's dynamic spatial model with WX terms
m_a_sdm = SDMPanelFE(
    formula=formula_a,
    data=df_a,
    W=W_a,
    unit_col="unit",
    time_col="time",
    model=1,
    priors={"rho_lower": -0.95, "rho_upper": 0.95},
)
# Quick-run settings to keep notebook execution practical on large panels.
idata_a_sdm = m_a_sdm.fit(draws=75, tune=75, chains=1, target_accept=0.9, random_seed=123, progressbar=False)

display(m_a_sdm.summary(round_to=4))
display(pd.DataFrame(m_a_sdm.diagnostics()))

Only 75 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [rho, beta, sigma]
Sampling 1 chain for 75 tune and 75 draw iterations (75 + 75 draws total) took 1 seconds.
There were 58 divergences after tuning. Increase `target_accept` or reparameterize.
The number of samples is too small to check convergence reliably.
arviz - WARNING - Shape validation failed: input_shape: (1, 75), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta[0],-0.8105,0.0059,-0.8255,-0.8070,0.0035,0.0027,2.9077,14.6463,NaN
beta[1],0.4173,0.0149,0.4093,0.4498,0.0086,0.0068,1.8300,12.4015,NaN
beta[2],0.0589,0.0030,0.0557,0.0662,0.0011,0.0016,9.3873,15.1847,NaN
beta[3],-0.0402,0.0360,-0.1193,-0.0198,0.0213,0.0165,1.4037,12.4015,NaN
beta[4],-0.2406,0.0390,-0.3310,-0.2189,0.0224,0.0182,1.5992,12.4015,NaN
beta[5],-0.4071,0.0217,-0.4555,-0.3943,0.0130,0.0098,1.4412,12.4015,NaN
beta[6],0.0003,0.0001,0.0001,0.0004,0.0000,0.0000,88.2923,34.4744,NaN
beta[7],-0.0742,0.0309,-0.1557,-0.0546,0.0186,0.0137,1.4430,12.6304,NaN
beta[8],-0.8594,0.0025,-0.8652,-0.8561,0.0006,0.0008,13.9452,31.9619,NaN
beta[9],0.0077,0.0032,0.0009,0.0110,0.0017,0.0013,6.5983,12.4015,NaN


,regression,heteroskedasticity,autocorrelation,outliers,panel
meth,rdiagnose_like,NaN,ljung_box_q,NaN,NaN
hatdi,"[0.0029549433704223658, 0.0014858566327801926,...",NaN,NaN,NaN,NaN
stdr,"[-0.42333496410315735, 2.1687154048008583, -1....",NaN,NaN,NaN,NaN
press,"[-1.525959331291521, 7.8058813957402355, -6.13...",NaN,NaN,NaN,NaN
pstat,121849.720955,NaN,NaN,NaN,NaN
stud,"[-0.42396181910936365, 2.1703284026206444, -1....",NaN,NaN,NaN,NaN
rstud,"[-0.4233164947585177, 2.16914246378613, -1.701...",NaN,NaN,NaN,NaN
dffit,"[-0.02304530074914291, 0.0836757125947251, -0....",NaN,NaN,NaN,NaN
cookd,"[7.610090676983349e-05, 0.0010013261415642016,...",NaN,NaN,NaN,NaN
resid,"[-1.521450207881987, 7.7942829750936795, -6.11...",NaN,NaN,NaN,NaN


In [7]:
# Alternative: spatial-error counterpart
m_a_sdem = SDEMPanelFE(
    formula=formula_a,
    data=df_a,
    W=W_a,
    unit_col="unit",
    time_col="time",
    model=1,
    priors={"lam_lower": -0.95, "lam_upper": 0.95},
)
idata_a_sdem = m_a_sdem.fit(draws=75, tune=75, chains=1, target_accept=0.9, random_seed=123, progressbar=False)

display(m_a_sdem.summary(round_to=4))
display(pd.DataFrame(m_a_sdem.diagnostics()))

Only 75 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [lam, beta, sigma]
Sampling 1 chain for 75 tune and 75 draw iterations (75 + 75 draws total) took 3386 seconds.
There were 58 divergences after tuning. Increase `target_accept` or reparameterize.
The number of samples is too small to check convergence reliably.
arviz - WARNING - Shape validation failed: input_shape: (1, 75), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta[0],-0.8367,0.0196,-0.8614,-0.7888,0.0103,0.0072,4.4511,13.5679,NaN
beta[1],0.3297,0.0081,0.3188,0.3459,0.0046,0.0022,3.8273,31.9477,NaN
beta[2],0.0128,0.0297,-0.0313,0.0449,0.0261,0.0035,1.4566,31.9477,NaN
beta[3],0.0488,0.0030,0.0453,0.0563,0.0004,0.0009,48.9828,31.9477,NaN
beta[4],-0.0974,0.0054,-0.1052,-0.0856,0.0015,0.0010,11.9526,15.7808,NaN
beta[5],0.0216,0.1033,-0.2409,0.0985,0.0738,0.0379,1.5775,12.4015,NaN
beta[6],0.0005,0.0001,0.0003,0.0006,0.0000,0.0000,20.9155,17.7478,NaN
beta[7],0.0240,0.0150,0.0031,0.0419,0.0059,0.0013,7.9434,30.7654,NaN
beta[8],-0.9321,0.0225,-0.9651,-0.9054,0.0101,0.0044,5.1782,17.3734,NaN
beta[9],0.0160,0.0229,-0.0125,0.0528,0.0177,0.0034,2.0465,23.6364,NaN


,regression,heteroskedasticity,autocorrelation,outliers,panel
meth,rdiagnose_like,NaN,ljung_box_q,NaN,NaN
hatdi,"[0.0029549433704223658, 0.0014858566327801926,...",NaN,NaN,NaN,NaN
stdr,"[-0.2629843534778958, 2.2145965042124036, -2.0...",NaN,NaN,NaN,NaN
press,"[-0.7817508967868724, 6.573454353421802, -6.17...",NaN,NaN,NaN,NaN
pstat,82962.923623,NaN,NaN,NaN,NaN
stud,"[-0.26337376865148243, 2.2162436264328167, -2....",NaN,NaN,NaN,NaN
rstud,"[-0.26297134167769615, 2.2150562876972186, -2....",NaN,NaN,NaN,NaN
dffit,"[-0.01431612926121676, 0.08544686040905689, -0...",NaN,NaN,NaN,NaN
cookd,"[2.9368473532989364e-05, 0.0010441421933584977...",NaN,NaN,NaN,NaN
resid,"[-0.7794408671570903, 6.563687142670492, -6.15...",NaN,NaN,NaN,NaN


## Spec B (MATLAB second block): no wages, full balanced sample N=3066, W from WBC.mat

In [ ]:
N1 = 3074
T1 = 7

check_cols_b = [COLS["lfp"], COLS["pc_urb"], COLS["pc_farm"], COLS["avg_educ"], COLS["pop_dens"]]
M1_b = np.sum(np.isnan(A[:, check_cols_b]), axis=1)
M2_b = M1_b.reshape(N1, T1, order="F")
select_b = (np.sum(M2_b > 0, axis=1) == 0)
N_b = int(np.sum(select_b))

W_b = row_normalize(WBC[np.ix_(select_b, select_b)])

selectfull_b = np.tile(select_b, 7)
y_b = A[selectfull_b, COLS["lfp"]]
x_b = A[selectfull_b][:, [COLS["pc_urb"], COLS["pc_farm"], COLS["avg_educ"], COLS["pop_dens"]]]

feature_names_b = ["pc_urb", "pc_farm", "avg_educ", "pop_dens"]
df_b, T_b = make_dynamic_panel_df(y_b, x_b, W_b, feature_names_b)

print("Spec B units:", N_b)
print("Spec B periods in transformed panel:", T_b - 1)
display(df_b.head())

In [ ]:
formula_b = "lfp ~ lfp_lag + w_lfp_lag + pc_urb + pc_farm + avg_educ + pop_dens"

# With WX terms (close to the MATLAB 'with WX' block)
m_b_sdm = SDMPanelFE(
    formula=formula_b,
    data=df_b,
    W=W_b,
    unit_col="unit",
    time_col="time",
    model=1,
    priors={"rho_lower": -0.95, "rho_upper": 0.95},
)
idata_b_sdm = m_b_sdm.fit(draws=75, tune=75, chains=1, target_accept=0.9, random_seed=456, progressbar=False)
display(m_b_sdm.summary(round_to=4))

In [ ]:
# No WX counterpart (MATLAB's 'NO WX' spirit)
m_b_sar = SARPanelFE(
    formula=formula_b,
    data=df_b,
    W=W_b,
    unit_col="unit",
    time_col="time",
    model=1,
    priors={"rho_lower": -0.95, "rho_upper": 0.95},
)
idata_b_sar = m_b_sar.fit(draws=75, tune=75, chains=1, target_accept=0.9, random_seed=456, progressbar=False)
display(m_b_sar.summary(round_to=4))

In [ ]:
# Compare key fit statistics and WAIC/LOO across candidate bayespecon specs
cmp = {
    "A_SDM": idata_a_sdm,
    "A_SDEM": idata_a_sdem,
    "B_SDM": idata_b_sdm,
    "B_SAR": idata_b_sar,
}

for name, idata in cmp.items():
    print(f"\n{name}")
    try:
        print("  WAIC:", az.waic(idata))
    except Exception as e:
        print("  WAIC unavailable:", e)
    try:
        print("  LOO :", az.loo(idata))
    except Exception as e:
        print("  LOO unavailable:", e)

## Notes on Replication Fidelity

- The MATLAB code estimates a **dynamic spatial panel with interactive fixed effects** via `SFactors` (QML with latent factors).
- `bayespecon` currently provides Bayesian spatial panel classes (`OLS/SAR/SEM/SDM/SDEM`) but not the exact `SFactors` estimator.
- This notebook replicates:
  1. sample selection logic,
  2. spatial-weights construction logic (6-NN and normalized `WBC`),
  3. dynamic-lag regressor construction,
  4. nearest available spatial panel model fits and diagnostics.

## Side-by-Side Comparison with Matlab_output.txt

This section parses the coefficient tables in `Matlab_output.txt` and compares them to posterior means from the fitted `bayespecon` models.

Notes:
- The original MATLAB model (`SFactors`) jointly estimates dynamic spatial lags and spatial error with latent factors.
- No single `bayespecon` model exactly matches that structure, so this is an approximate coefficient comparison.

In [ ]:
import re

text = (ref_dir / 'Matlab_output.txt').read_text(errors='ignore')
sections = text.split('DSPD with PCs')

coef_pattern = re.compile(
    r'^(LFP\(t-1\)|W\*LFP\(t-1\)|pc_urb|pc_farm|avg_educ|pop_dens|wages|'
    r'W\*pc_urb|W\*pc_farm|W\*avg_educ|W\*pop_dens|W\*wages|W\*LFP\(t\)|W\*error)\s+'
    r'([-+]?[0-9]*\.?[0-9]+)\s+'
    r'([-+]?[0-9]*\.?[0-9]+)\s+'
    r'([-+]?[0-9]*\.?[0-9]+)$'
)

def parse_table(sec):
    rows = []
    for ln in sec.splitlines():
        m = coef_pattern.match(ln.strip())
        if m:
            rows.append((m.group(1), float(m.group(2)), float(m.group(3)), float(m.group(4))))
    return pd.DataFrame(rows, columns=['term', 'matlab_coef', 'matlab_sdev', 'matlab_t'])

# Tables in order shown by MATLAB output
tab1 = parse_table(sections[1])  # wages included
tab2 = parse_table(sections[2])  # no wages, with WX
tab3 = parse_table(sections[3])  # no WX


def pmean(idata, var, idx=None):
    arr = idata.posterior[var].mean(('chain', 'draw')).to_numpy()
    if idx is None:
        return float(arr)
    return float(arr[idx])

# Approximate mapping for Spec A (A_SDM)
map_a = {
    'LFP(t-1)': ('beta', 0),
    'W*LFP(t-1)': ('beta', 1),
    'pc_urb': ('beta', 2),
    'pc_farm': ('beta', 3),
    'avg_educ': ('beta', 4),
    'pop_dens': ('beta', 5),
    'wages': ('beta', 6),
    'W*pc_urb': ('beta', 9),
    'W*pc_farm': ('beta', 10),
    'W*avg_educ': ('beta', 11),
    'W*pop_dens': ('beta', 12),
    'W*wages': ('beta', 13),
    'W*LFP(t)': ('rho', None),
    # 'W*error' has no exact counterpart in SDMPanelFE
}

# Approximate mapping for Spec B with WX (B_SDM)
map_b_wx = {
    'LFP(t-1)': ('beta', 0),
    'W*LFP(t-1)': ('beta', 1),
    'pc_urb': ('beta', 2),
    'pc_farm': ('beta', 3),
    'avg_educ': ('beta', 4),
    'pop_dens': ('beta', 5),
    'W*pc_urb': ('beta', 8),
    'W*pc_farm': ('beta', 9),
    'W*avg_educ': ('beta', 10),
    'W*pop_dens': ('beta', 11),
    'W*LFP(t)': ('rho', None),
}

# Approximate mapping for Spec B no-WX (B_SAR)
map_b_nowx = {
    'LFP(t-1)': ('beta', 0),
    'W*LFP(t-1)': ('beta', 1),
    'pc_urb': ('beta', 2),
    'pc_farm': ('beta', 3),
    'avg_educ': ('beta', 4),
    'pop_dens': ('beta', 5),
    'W*LFP(t)': ('rho', None),
}


def compare_table(tab, idata, mapping, label):
    out = tab.copy()
    out['bayespecon_coef'] = np.nan
    for i, term in enumerate(out['term']):
        if term in mapping:
            v, idx = mapping[term]
            out.loc[i, 'bayespecon_coef'] = pmean(idata, v, idx)
    out['diff_bayes_minus_matlab'] = out['bayespecon_coef'] - out['matlab_coef']
    print(f'\n{label}')
    display(out)


compare_table(tab1, idata_a_sdm, map_a, 'Spec A: MATLAB table 1 vs bayespecon A_SDM')
compare_table(tab2, idata_b_sdm, map_b_wx, 'Spec B (with WX): MATLAB table 2 vs bayespecon B_SDM')
compare_table(tab3, idata_b_sar, map_b_nowx, 'Spec B (no WX): MATLAB table 3 vs bayespecon B_SAR')